# MGS-28 : Bare Bones PSO en Rust — le PSO sans vitesse, et ses deux ancrages

**Navigation** : [<< MGS-27 (FBI vs mealpy)](MGS-27-ForensicBasedInvestigation-vs-Mealpy.ipynb) | [Index](README.md)

**Kernel** : Rust (evcxr) — trois bras dans la même exécution, sans pont PythonNet

***

## Introduction

MGS-27 a refermé la question FBI sur la paire qui prétendait le moins à la différence : le port C# dérivé battait sa référence mealpy des deux mains. Cette paire descend d'un cran encore dans la prétention — et change de nature : le **Bare Bones PSO** de Kennedy (2003) est le PSO dépouillé de toute sa mécanique, ni inertie *w*, ni coefficients *c1*/*c2* : chaque gène de la nouvelle particule est un simple **tirage gaussien** autour du milieu entre le meilleur personnel et le meilleur global. Deux conséquences structurelles gouvernent ce protocole :

1. **Aucun PSO de mealpy 3.0.2 n'est bare-bones** — les sept variantes du module (`OriginalPSO`, `P_PSO`, `C_PSO`, `CL_PSO`, `AIW_PSO`, `HPSO_TVAC`, `LDW_PSO`) conservent toutes une **vitesse explicite** et un couple de coefficients de vitesse — constantes *w*/*c1*/*c2* (ou leurs variantes `w_min`/`w_max`/`ci`/`cf`/`c_local`) pour six d'entre elles, coefficients sin/cos dynamiques pour `P_PSO` ; en prendre une pour jumeau serait mesurer un **PSO classique déguisé** en bare-bones. Le jumeau mealpy est donc un *subclass* de `Optimizer` — le point d'extension natif de la bibliothèque — qui **pinne la formule de Kennedy** dans le harnais mealpy (graine, comptage d'évaluations, correction de bornes). Le port Rust écrit directement cet opérateur et confie les tirages à un `StdRng` semé explicitement avant la création de la population.
2. **Le portage MGS est ancré sur la position courante, pas sur le pbest** — le framework géométrique n'expose pas de mémoire *pbest_i* par particule, et l'écart est documenté *in-source* dans `BareBonesParticleSwarm.cs`. Pour distinguer l'écart **noyau** de l'écart dit **d'ancrage**, le bench croise donc **trois bras** en Rust : le noyau (position courante, sélection best-N), le port aligné (position courante, sélection gloutonne par particule), et le port littéral de Kennedy 2003 (`pbest_i`, remplacement inconditionnel).

La grille, la représentation R1, le budget (population 50, 160 générations) et les graines {0, 1, 7, 42} sont ceux des paires précédentes : les médianes se lisent dans la même colonne du tableau croisé de la série.

***

In [2]:
:dep ndarray = "=0.17.2"
:dep rand = "=0.9.2"
:dep rand_distr = "=0.5.1"
:dep serde_json = { version = "=1.0.145", features = ["float_roundtrip"] }

// === MGS-28 : socle commun — grille de référence, fonction de coût ===
// Même socle que MGS-22/23 : la représentation R1 (continu + arrondi) est le substrat du bench.
// Références DLL du carnet .NET d'origine, non chargées en Rust :
// r "../../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
// r "../../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
// r "../../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
{
    use ndarray::Array2;

    // Grille facile Easy[0] de Sudoku_Easy51.txt — la MÊME que MGS-21/22/23/25/27.
    const PUZZLE_LINE_28: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const REFERENCE: (usize, usize, usize) = (36, 45, 36);

    fn parse_puzzle_28() -> Result<Array2<i64>, &'static str> {
        if PUZZLE_LINE_28.len() != 81 || !PUZZLE_LINE_28.bytes().all(|b| b.is_ascii_digit()) {
            return Err("Easy[0] doit contenir exactement 81 chiffres ASCII");
        }
        let digits = PUZZLE_LINE_28.bytes().map(|b| i64::from(b - b'0')).collect();
        Array2::from_shape_vec((9, 9), digits).map_err(|_| "forme de grille invalide")
    }

    // Fonction de coût du bench : conflits totaux (lignes + colonnes + blocs) sur grille PLEINE.
    // Renvoie 0 ssi résolu. La seconde implémentation reprend le même comptage.
    fn count_conflicts_28(grid: &Array2<i64>) -> usize {
        fn duplicate(seen: &mut u16, digit: i64) -> usize {
            let bit = 1_u16 << (digit as u32);
            let repeated = (*seen & bit) != 0;
            *seen |= bit;
            usize::from(repeated)
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let (mut row, mut col, mut block) = (0, 0, 0);
            for j in 0..9 {
                conflicts += duplicate(&mut row, grid[[i, j]]);
                conflicts += duplicate(&mut col, grid[[j, i]]);
                conflicts += duplicate(&mut block, grid[[3 * (i / 3) + j / 3, 3 * (i % 3) + j % 3]]);
            }
        }
        conflicts
    }

    fn count_empty_28(puzzle: &Array2<i64>) -> usize {
        puzzle.iter().filter(|&&digit| digit == 0).count()
    }

    fn empty_cells_28(puzzle: &Array2<i64>) -> Vec<(usize, usize)> {
        puzzle.indexed_iter().filter_map(|(pos, &digit)| (digit == 0).then_some(pos)).collect()
    }

    // Décodage R1 : arrondi + clamp vers 1..9 sur les cellules vides, ordre de lecture.
    fn decode_r1_28(genes: &[f64]) -> Result<Array2<i64>, &'static str> {
        let mut grid = parse_puzzle_28()?;
        let empty = empty_cells_28(&grid);
        let used = genes.get(..empty.len()).ok_or("vecteur R1 trop court")?;
        for ((r, c), &gene) in empty.into_iter().zip(used) {
            if !gene.is_finite() {
                return Err("gène R1 non fini");
            }
            grid[[r, c]] = gene.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        Ok(grid)
    }

    // LCG des témoins : arithmétique non signée modulo 2^32, puis projection sur [1, 10).
    fn lcg_vector_28(mut state: u32, n: usize) -> Vec<f64> {
        (0..n).map(|_| {
            state = state.wrapping_mul(1_664_525).wrapping_add(1_013_904_223);
            1.0 + (f64::from(state) / 4_294_967_296.0) * 9.0
        }).collect()
    }

    let result = (|| -> Result<(), &'static str> {
        let puzzle = parse_puzzle_28()?;
        let empty = count_empty_28(&puzzle);
        let fixed = puzzle.len() - empty;
        let genes = empty_cells_28(&puzzle).len();
        println!("Grille de référence : {empty} cellules vides, {fixed} indices fixes, {genes} gènes R1.");
        let measured = (empty, fixed, genes);
        if measured != REFERENCE {
            println!("Dimensions : calculé {measured:?} | attendu {REFERENCE:?} -> ÉCART");
            return Err("dimensions R1 différentes de la référence");
        }
        // Premier témoin du contrôle croisé : le coût est recalculé, pas fourni au décodeur.
        let probe_cost = count_conflicts_28(&decode_r1_28(&lcg_vector_28(1, 51))?);
        if probe_cost != 67 {
            println!("Témoin LCG 1 : calculé {probe_cost} | attendu 67 -> ÉCART");
            return Err("coût du premier témoin différent de la référence");
        }
        Ok(())
    })();
    if let Err(error) = result {
        eprintln!("Erreur du socle Sudoku : {error}");
    }
};

Grille de référence : 36 cellules vides, 45 indices fixes, 36 gènes R1.


**Lecture.** Le socle est posé, identique à MGS-23 au nom près — c'est voulu : la comparabilité
des paires de la série repose sur un protocole strictement apparié (même grille Easy[0], même
représentation R1 continue à 36 gènes, même fonction de coût comptée en conflits, renvoyant 0
ssi la grille est résolue). Les gènes remplissent les cases vides dans l'ordre de lecture :
l'arrondi au pair à mi-distance précède le bornage entre 1 et 9. Le contrôle Rust qui suivra
réimplémente *à l'identique* le comptage des répétitions en lignes, colonnes et blocs et le
vérifie sur des vecteurs témoins déterministes : la comparaison des bras ne porte pas sur
l'échelle de la fitness.

In [3]:
// === Noyau BBPSO : chromosome R1, fitness instrumentée, ancre courante et sélection best-N ===
// Kennedy (2003) "Bare Bones" : chaque gène est échantillonné N((ancre + gbest)/2, |gbest - ancre|).
// Ni inertie w ni c1/c2 — les constantes de Clerc (w=0,7298 ; c1=c2=1,49618) NE S'APPLIQUENT PAS
// à ce composé : un BBPSO qui les porterait serait un PSO classique déguisé.
{
    use ndarray::Array2;
    use rand::{rngs::StdRng, Rng, SeedableRng};
    use rand_distr::{Distribution, Normal};
    use std::time::Instant;

    const PUZZLE_LINE_28: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const POPULATION: usize = 50;
    const GENERATIONS: usize = 160;
    const REFERENCE_EVALS: usize = 8_000;
    const REFERENCE_CONFLICTS: usize = 29;
    const REFERENCE_CHECKPOINTS: [usize; 4] = [29; 4];
    // Même bande que pour le banc : les suites StdRng et .NET diffèrent à graine égale.
    const CONFLICT_TOLERANCE: usize = 12;

    #[derive(Debug)]
    enum SolveError {
        InvalidConfig,
        InvalidPuzzle,
        InvalidGenes,
        InvalidNormal,
        EmptyPopulation,
    }

    struct SudokuR1Chromosome28 {
        genes: Vec<f64>,
    }

    struct Evaluated {
        chromosome: SudokuR1Chromosome28,
        conflicts: usize,
    }

    struct Genome {
        best: Evaluated,
        evals: usize,
        checkpoints: [usize; 4],
    }

    fn parse_puzzle_28() -> Result<Array2<i64>, SolveError> {
        if PUZZLE_LINE_28.len() != 81 || !PUZZLE_LINE_28.bytes().all(|b| b.is_ascii_digit()) {
            return Err(SolveError::InvalidPuzzle);
        }
        let digits = PUZZLE_LINE_28.bytes().map(|b| i64::from(b - b'0')).collect();
        Array2::from_shape_vec((9, 9), digits).map_err(|_| SolveError::InvalidPuzzle)
    }

    fn decode_r1_28(puzzle: &Array2<i64>, empty: &[(usize, usize)], genes: &[f64]) -> Result<Array2<i64>, SolveError> {
        let mut grid = puzzle.clone();
        let used = genes.get(..empty.len()).ok_or(SolveError::InvalidGenes)?;
        for (&(r, c), &gene) in empty.iter().zip(used) {
            if !gene.is_finite() {
                return Err(SolveError::InvalidGenes);
            }
            grid[[r, c]] = gene.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        Ok(grid)
    }

    fn count_conflicts_28(grid: &Array2<i64>) -> usize {
        fn duplicate(seen: &mut u16, digit: i64) -> usize {
            let bit = 1_u16 << (digit as u32);
            let repeated = (*seen & bit) != 0;
            *seen |= bit;
            usize::from(repeated)
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let (mut row, mut col, mut block) = (0, 0, 0);
            for j in 0..9 {
                conflicts += duplicate(&mut row, grid[[i, j]]);
                conflicts += duplicate(&mut col, grid[[j, i]]);
                conflicts += duplicate(&mut block, grid[[3 * (i / 3) + j / 3, 3 * (i % 3) + j % 3]]);
            }
        }
        conflicts
    }

    // Fitness instrumentée : chaque évaluation est comptée — le budget se mesure, il ne se suppose pas.
    struct Fitness<'a> {
        puzzle: &'a Array2<i64>,
        empty: &'a [(usize, usize)],
        evals: usize,
    }

    impl Fitness<'_> {
        fn evaluate(&mut self, chromosome: SudokuR1Chromosome28) -> Result<Evaluated, SolveError> {
            let grid = decode_r1_28(self.puzzle, self.empty, &chromosome.genes)?;
            let conflicts = count_conflicts_28(&grid);
            self.evals += 1;
            Ok(Evaluated { chromosome, conflicts })
        }
    }

    fn sample_normal(rng: &mut StdRng, mean: f64, std: f64) -> Result<f64, SolveError> {
        if std == 0.0 {
            // Loi dégénérée : la coordonnée reste à la moyenne, sans consommer de tirage.
            return Ok(mean);
        }
        let normal = Normal::new(mean, std).map_err(|_| SolveError::InvalidNormal)?;
        Ok(normal.sample(rng))
    }

    // Trajectoire best-so-far : conflits du meilleur APRÈS chaque génération (boucle Rust).
    fn run_kernel(seed: u64, pop_size: usize, generations: usize, std_floor: f64) -> Result<Genome, SolveError> {
        if pop_size < 2 || generations < 4 || pop_size.checked_mul(generations).is_none()
            || !std_floor.is_finite() || std_floor < 0.0
        {
            return Err(SolveError::InvalidConfig);
        }
        let puzzle = parse_puzzle_28()?;
        let empty: Vec<_> = puzzle.indexed_iter()
            .filter_map(|(pos, &digit)| (digit == 0).then_some(pos)).collect();
        let mut fitness = Fitness { puzzle: &puzzle, empty: &empty, evals: 0 };

        // Seeding AVANT création de population : le RNG est consommé par la création
        // de chaque individu initial (leçon du carnet MGS-21).
        let mut rng = StdRng::seed_from_u64(seed);
        let mut population = Vec::with_capacity(2 * pop_size);
        for _ in 0..pop_size {
            let genes = (0..empty.len()).map(|_| rng.random_range(1.0..10.0)).collect();
            population.push(fitness.evaluate(SudokuR1Chromosome28 { genes })?);
        }
        // Tri stable : à coût égal, les parents précèdent les enfants, dans leur ordre existant.
        population.sort_by_key(|particle| particle.conflicts);

        // Checkpoints 25/50/75/100 % du budget : trajectoire indexée après la génération correspondante.
        let quarters = [generations / 4, generations / 2,
            3 * (generations / 4) + 3 * (generations % 4) / 4, generations];
        let mut checkpoints = [0; 4];
        // L'initialisation est la génération 1 du noyau : 50 + 159 × 50 = 8 000 évaluations.
        for generation in 1..=generations {
            if generation > 1 {
                // gbest est figé pour la génération : tous les enfants lisent la même élite.
                let gbest = &population.first().ok_or(SolveError::EmptyPopulation)?.chromosome.genes;
                let mut children = Vec::with_capacity(pop_size);
                for parent in &population {
                    let mut genes = Vec::with_capacity(gbest.len());
                    for (&x, &g) in parent.chromosome.genes.iter().zip(gbest) {
                        let mean = (x + g) / 2.0;
                        let std = (g - x).abs().max(std_floor);
                        genes.push(sample_normal(&mut rng, mean, std)?.clamp(1.0, 10.0));
                    }
                    children.push(fitness.evaluate(SudokuR1Chromosome28 { genes })?);
                }
                population.extend(children);
                population.sort_by_key(|particle| particle.conflicts);
                population.truncate(pop_size);
            }
            if let Some(q) = quarters.iter().position(|&mark| mark == generation) {
                checkpoints[q] = population.first().ok_or(SolveError::EmptyPopulation)?.conflicts;
            }
        }
        // Épuiser normalement le budget rend l'incumbent ; une erreur seule rend Err.
        let best = population.into_iter().next().ok_or(SolveError::EmptyPopulation)?;
        Ok(Genome { best, evals: fitness.evals, checkpoints })
    }

    let result = (|| -> Result<(), SolveError> {
        // Échauffement (course jetée), puis course témoin graine 7.
        let _warmup = run_kernel(123, POPULATION, 10, 0.0)?;
        let start = Instant::now();
        let demo = run_kernel(7, POPULATION, GENERATIONS, 0.0)?;
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let checkpoints = demo.checkpoints.map(|c| c.to_string()).join("/");
        println!("Noyau Rust BBPSO (graine 7, témoin) : {} conflits, {} évaluations, {ms:.0} ms, checkpoints 25/50/75/100 % = {checkpoints}.", demo.best.conflicts, demo.evals);

        let budget_ok = demo.evals == REFERENCE_EVALS && demo.evals == POPULATION * GENERATIONS;
        let elite_ok = demo.checkpoints.windows(2).all(|pair| pair[0] >= pair[1])
            && demo.checkpoints[3] == demo.best.conflicts;
        println!("Budget : calculé {} | attendu {REFERENCE_EVALS} -> {}", demo.evals,
            if budget_ok { "CONFORME" } else { "ÉCART" });
        println!("Élitisme : checkpoints non croissants et dernier = score final -> {}",
            if elite_ok { "CONFORME" } else { "ÉCART" });
        let delta = demo.best.conflicts.abs_diff(REFERENCE_CONFLICTS);
        let cp_delta = std::array::from_fn::<_, 4, _>(|i| demo.checkpoints[i].abs_diff(REFERENCE_CHECKPOINTS[i]));
        let band_ok = delta <= CONFLICT_TOLERANCE && cp_delta.iter().all(|&d| d <= CONFLICT_TOLERANCE);
        println!("Référence .NET : {REFERENCE_CONFLICTS} conflits, checkpoints {REFERENCE_CHECKPOINTS:?} ; écarts absolus : final {delta}, checkpoints {cp_delta:?} (tolérance {CONFLICT_TOLERANCE}) -> {}",
            if band_ok { "CONFORME" } else { "ÉCART" });
        Ok(())
    })();
    if let Err(error) = result {
        eprintln!("Erreur du noyau BBPSO : {error:?}");
    }
};

Noyau Rust BBPSO (graine 7, témoin) : 24 conflits, 8000 évaluations, 19 ms, checkpoints 25/50/75/100 % = 24/24/24/24.


Budget : calculé 8000 | attendu 8000 -> CONFORME


Élitisme : checkpoints non croissants et dernier = score final -> CONFORME


Référence .NET : 29 conflits, checkpoints [29, 29, 29, 29] ; écarts absolus : final 5, checkpoints [5, 5, 5, 5] (tolérance 12) -> CONFORME


**Lecture.** Le noyau reprend le composé `BareBonesParticleSwarm`, transcription du **Bare Bones
Particle Swarms** de Kennedy (SIS 2003) dans la grammaire géométrique MGS : l'opérateur
géométrique à deux parents lit `[position courante, gbest]` et tire chaque gène
dans `N((x_i + gbest)/2, |gbest - x_i|)` — la formule est **pinnée in-source**
(`sample_normal`, `rand_distr::Normal` sur `StdRng`). Trois propriétés à lire dans le
portage : (1) **pas de mémoire pbest** — l'ancre personnelle est la position *courante*
(variante « gbest-ancrée », écart documenté dans le fichier source, la cellule Formules le
croise) ; (2) **gel au meilleur global** — quand une particule *est* le gbest, l'écart-type
s'annule et son tirage retombe sur elle-même : l'élite ne bouge pas, par construction ; (3)
**sélection élitiste** (best-N parents+enfants) —
le garde-fou qui empêche un mauvais tirage gaussien de détruire l'incumbent. La trajectoire
best-so-far est relevée à la fin des générations correspondant aux checkpoints
25/50/75/100 % du budget pour la classification de la forme d'écart.

La graine est la même que dans le témoin .NET, pas la suite de tirages : `StdRng` ne reproduit
pas le générateur de MGS. Le budget est comparé à l'égalité ; le score et les checkpoints
calculés sont comparés à la référence .NET sous la bande annoncée, sans comparaison des durées.

In [4]:
// === Contrôle croisé : deux implémentations Rust dans le même kernel, la même exécution ===
// Dans le carnet d'origine, le CPython vient de PYTHONNET_PYDLL s'il est défini, sinon du premier interpréteur
// installé qui importe mealpy (Config/PythonNetLoader.cs : Windows, Linux, macOS). Aucun interpréteur Python ici.
// Le problème de contrôle : décodage + coût réimplémentés à l'identique ; le compteur d'évals reste dans les moteurs.
// Le JUMEAU BARE-BONES d'origine est un subclass d'Optimizer (point d'extension natif mealpy) épinglant
// la formule de Kennedy 2003 — aucune des 7 variantes PSO de mealpy 3.0.2 n'est bare-bones.
{
    use ndarray::Array2;

    const PUZZLE_LINE_28: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const EXPECTED: [usize; 3] = [67, 71, 60];

    fn lcg_vector_28(mut state: u32, n: usize) -> Vec<f64> {
        (0..n).map(|_| {
            state = state.wrapping_mul(1_664_525).wrapping_add(1_013_904_223);
            1.0 + (f64::from(state) / 4_294_967_296.0) * 9.0
        }).collect()
    }

    // Voie A : grille ndarray, arrondi au pair natif et masques de présence par unité.
    fn decode_r1_28(genes: &[f64]) -> Result<Array2<i64>, &'static str> {
        if PUZZLE_LINE_28.len() != 81 || !PUZZLE_LINE_28.bytes().all(|b| b.is_ascii_digit()) {
            return Err("Easy[0] doit contenir exactement 81 chiffres ASCII");
        }
        let digits = PUZZLE_LINE_28.bytes().map(|b| i64::from(b - b'0')).collect();
        let mut grid = Array2::from_shape_vec((9, 9), digits).map_err(|_| "forme de grille invalide")?;
        let mut genes = genes.iter().copied();
        for digit in &mut grid {
            if *digit == 0 {
                let gene = genes.next().ok_or("vecteur R1 trop court")?;
                if !gene.is_finite() {
                    return Err("gène R1 non fini");
                }
                *digit = gene.round_ties_even().clamp(1.0, 9.0) as i64;
            }
        }
        Ok(grid)
    }

    fn count_conflicts_28(grid: &Array2<i64>) -> usize {
        fn duplicate(seen: &mut u16, digit: i64) -> usize {
            let bit = 1_u16 << (digit as u32);
            let repeated = (*seen & bit) != 0;
            *seen |= bit;
            usize::from(repeated)
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let (mut row, mut col, mut block) = (0, 0, 0);
            for j in 0..9 {
                conflicts += duplicate(&mut row, grid[[i, j]]);
                conflicts += duplicate(&mut col, grid[[j, i]]);
                conflicts += duplicate(&mut block, grid[[3 * (i / 3) + j / 3, 3 * (i % 3) + j % 3]]);
            }
        }
        conflicts
    }

    // Voie B : grille plate et arrondi au pair explicite, sans appeler le décodeur A.
    fn decode_flat_28(genes: &[f64]) -> Result<[usize; 81], &'static str> {
        let bytes = PUZZLE_LINE_28.as_bytes();
        if bytes.len() != 81 {
            return Err("longueur de grille invalide");
        }
        let mut grid = [0; 81];
        let mut k = 0;
        for (slot, &byte) in grid.iter_mut().zip(bytes) {
            *slot = match byte {
                b'1'..=b'9' => usize::from(byte - b'0'),
                b'0' => {
                    let &gene = genes.get(k).ok_or("vecteur R1 trop court")?;
                    k += 1;
                    if !gene.is_finite() {
                        return Err("gène R1 non fini");
                    }
                    // Borner avant l'arrondi donne le même chiffre, les deux bornes étant entières.
                    let bounded = gene.clamp(1.0, 9.0);
                    let lower = bounded.floor() as usize;
                    let fraction = bounded - lower as f64;
                    lower + usize::from(fraction > 0.5 || (fraction == 0.5 && lower % 2 == 1))
                }
                _ => return Err("indice de grille invalide"),
            };
        }
        Ok(grid)
    }

    fn count_histograms_28(grid: &[usize; 81]) -> usize {
        let mut counts = [[0_usize; 10]; 27];
        for (i, &digit) in grid.iter().enumerate() {
            let (r, c) = (i / 9, i % 9);
            counts[r][digit] += 1;
            counts[9 + c][digit] += 1;
            counts[18 + 3 * (r / 3) + c / 3][digit] += 1;
        }
        counts.iter().flatten().map(|&n| n.saturating_sub(1)).sum()
    }

    // --- Sanity check : la fonction de coût est-elle la MÊME des deux côtés ? ---
    // 3 vecteurs témoins DÉTERMINISTES (LCG écrit à la main), décodés et costés des deux côtés.
    let result = (|| -> Result<(), String> {
        let witnesses = [1, 2, 3].map(|seed| lcg_vector_28(seed, 51));
        // On garde les 51 composantes du témoin ; les décodeurs ne lisent que les 36 utiles.
        let json = serde_json::to_string(&witnesses).map_err(|e| e.to_string())?;
        let received: [Vec<f64>; 3] = serde_json::from_str(&json).map_err(|e| e.to_string())?;
        if received != witnesses {
            return Err("les vecteurs diffèrent après l'aller-retour JSON".to_string());
        }
        let mut a = [0; 3];
        let mut b = [0; 3];
        for i in 0..witnesses.len() {
            a[i] = count_conflicts_28(&decode_r1_28(&witnesses[i])?);
            b[i] = count_histograms_28(&decode_flat_28(&received[i])?);
        }
        let agrees = a == b && a == EXPECTED;
        let show = |costs: [usize; 3]| costs.map(|c| c.to_string()).join(",");
        println!("Sanity check cout : Rust A {} | Rust B {} -> {}",
            show(a), show(b), if agrees { "IDENTIQUE" } else { "ÉCART" });
        if !agrees {
            return Err(format!("référence exacte pour les graines 1, 2, 3 : {}", show(EXPECTED)));
        }
        Ok(())
    })();
    if let Err(error) = result {
        eprintln!("Échec du contrôle de coût : {error}");
    }
};

Sanity check cout : Rust A 67,71,60 | Rust B 67,71,60 -> IDENTIQUE


**Lecture.** Le contrôle croisé porte tout le bench : la seconde fonction de coût Rust
redonne *exactement* le comptage de la première sur les vecteurs témoins déterministes.
Cet accord valide l'échelle du coût sur ces témoins ; c'est un contrôle plus faible que
l'accord entre C# et Python du carnet d'origine. Le point structurant du carnet d'origine
est le **subclass `OriginalBBPSO`** : mealpy n'offre aucun PSO bare-bones (six des sept
variantes du module exposent explicitement *w*/*c1*/*c2* — ou leurs variantes
`w_min`/`w_max`/`ci`/`cf`/`c_local` — et la septième, `P_PSO`, porte des coefficients
sin/cos dynamiques sur la même mise à jour de vitesse), et le point d'extension officiel de la
bibliothèque est précisément la classe `Optimizer`. En Rust, les deux variantes partagent
un harnais Rust (générateur `StdRng` semé avant la population, comptage d'évaluations,
correction de bornes, suivi `gbest`) en pinnant la formule de Kennedy.
Les deux variantes du jumeau (port littéral et port aligné) ne diffèrent **que** par l'ancre et la règle
de remplacement — tout le reste (harnais, budget, graine) est identique par construction.

In [5]:
// === CELLULE FORMULES : les deux jumeaux AVANT le croisement ===
// Objectif : détecter un PSO classique déguisé AVANT de mesurer, puis trancher les divergences
// d'ancre et de sélection pour que l'écart mesuré s'interprète (noyau vs portage), pas le subir.
{
    use rand::{rngs::StdRng, SeedableRng};
    use rand_distr::{Distribution, Normal};

    // 1) La détection Clerc : aucun PSO mealpy 3.0.2 n'est bare-bones — six portent w/c1/c2
    // (ou leurs variantes w_min/w_max/ci/cf/c_local) ; P_PSO porte des coefficients
    // sin/cos dynamiques sur la même mise à jour de vitesse.
    // Données documentaires transcrites de la sortie d'origine, pas des valeurs par défaut Rust.
    let catalogue = [
        ("OriginalPSO", "[('c1', 2.05), ('c2', 2.05), ('w', 0.4)]"),
        ("P_PSO", "[]"),
        ("C_PSO", "[('c1', 2.05), ('c2', 2.05), ('w_max', 0.9), ('w_min', 0.4)]"),
        ("CL_PSO", "[('c_local', 1.2), ('w_max', 0.9), ('w_min', 0.4)]"),
        ("AIW_PSO", "[('c1', 2.05), ('c2', 2.05)]"),
        ("HPSO_TVAC", "[('cf', 0.1), ('ci', 0.5)]"),
        ("LDW_PSO", "[('c1', 2.05), ('c2', 2.05), ('w_max', 0.9), ('w_min', 0.4)]"),
    ];
    println!("Catalogue PSO mealpy 3.0.2 (référence du carnet d'origine ; aucun bare-bones natif) :");
    for (name, parameters) in catalogue {
        println!("{name:<10} {parameters}");
    }
    println!("Variantes BBPSO utilisées en Rust : noyau, port-aligné, port-littéral (implémentations explicites).");

    // 2) La propriété de gel du bare-bones : std = 0 => tirage déterministe = la moyenne.
    fn sample_normal(rng: &mut StdRng, mean: f64, std: f64) -> Result<f64, &'static str> {
        if std == 0.0 {
            return Ok(mean);
        }
        let normal = Normal::new(mean, std).map_err(|_| "paramètres gaussiens invalides")?;
        Ok(normal.sample(rng))
    }

    let freeze = (|| -> Result<(), &'static str> {
        let loc = [3.5, 7.25];
        const EXPECTED: [[f64; 2]; 4] = [[3.5, 7.25]; 4];
        let mut rng = StdRng::seed_from_u64(0);
        let mut draws = [[0.0; 2]; 4];
        for draw in &mut draws {
            for (coordinate, mean) in draw.iter_mut().zip(loc) {
                *coordinate = sample_normal(&mut rng, mean, 0.0)?;
            }
        }
        println!("Propriété de gel (Rust) : normal(loc, std=0) -> {draws:?}");
        let ok = draws == EXPECTED;
        println!("Gel : calculé {draws:?} | attendu {EXPECTED:?} -> {}", if ok { "CONFORME" } else { "ÉCART" });
        if ok { Ok(()) } else { Err("le tirage dégénéré diffère de la moyenne") }
    })();
    if let Err(error) = freeze {
        eprintln!("Échec du contrôle de gel : {error}");
    }

    println!(r#"
Formules épinglées des jumeaux :
  Noyau — reprise du composé MGS BareBonesParticleSwarm :
      x_new ~ N((x_i + gbest)/2, |gbest - x_i|)   [ancre = position COURANTE x_i]
      sélection : best-N parents+enfants (glouton sur le pool)
  Port-littéral — Kennedy 2003 :
      x_new ~ N((pbest_i + gbest)/2, |gbest - pbest_i|)   [ancre = pbest_i]
      sélection : remplacement INCONDITIONNEL (la mémoire vit dans pbest/gbest)
  Port-aligné — même ancre que le noyau :
      x_new ~ N((x_i + gbest)/2, |gbest - x_i|)   [ancre = position courante]
      sélection : glouton par particule (remplace le parent seulement si l'enfant l'améliore)

TRANCHE : deux divergences documentées — (1) l'ANCRE (courante vs pbest), (2) la SÉLECTION
(best-N sur le pool vs glouton par particule vs remplacement inconditionnel).
  noyau - port-aligné         : ÉCART NOYAU (même ancre, sélection et budget différents)
  port-aligné - port-littéral : ÉCART D'ANCRAGE (même harnais, ancre et remplacement différents)
  noyau - port-littéral       : ÉCART BRUT
"#);
};

Catalogue PSO mealpy 3.0.2 (référence du carnet d'origine ; aucun bare-bones natif) :


OriginalPSO [('c1', 2.05), ('c2', 2.05), ('w', 0.4)]


P_PSO      []


C_PSO      [('c1', 2.05), ('c2', 2.05), ('w_max', 0.9), ('w_min', 0.4)]


CL_PSO     [('c_local', 1.2), ('w_max', 0.9), ('w_min', 0.4)]


AIW_PSO    [('c1', 2.05), ('c2', 2.05)]


... (16 outputs left out)

(best-N sur le pool vs glouton par particule vs remplacement inconditionnel).


  noyau - port-aligné         : ÉCART NOYAU (même ancre, sélection et budget différents)


  port-aligné - port-littéral : ÉCART D'ANCRAGE (même harnais, ancre et remplacement différents)


  noyau - port-littéral       : ÉCART BRUT


**Lecture.** La tranche est posée *avant* le croisement : sans elle, l'écart brut sur cette paire serait **ininterprétable** — il mêlerait la sélection (glouton sur le pool, glouton par particule ou remplacement inconditionnel), l'ancre (courante ou *pbest*) et le décompte de l'initialisation. Le contraste « noyau » conserve l'ancre mais change la sélection et le décompte de l'initialisation ; le contraste « ancrage » conserve le harnais mais change aussi le remplacement.

Le catalogue de la cellule établit la **détection Clerc** : six des sept PSO mealpy exposent explicitement les constantes de vitesse (*w*/*c1*/*c2* ou leurs variantes), et le septième, `P_PSO`, porte des coefficients sin/cos dynamiques sur la même mise à jour de vitesse — **aucun** ne prétend au bare-bones (la ligne `P_PSO` est la seule à ne lister aucun de ces paramètres). Le carnet d'origine utilise donc un subclass ; ici, les bras Rust écrivent directement l'opérateur. La propriété de gel (écart-type nul ⇒ tirage déterministe) est le comportement littéral de Kennedy : si l'ancre coïncide avec le gbest, elle ne bouge pas, et c'est une **propriété**, pas un bug à corriger — l'Exercice 2 en mesurera le prix.

In [6]:
// === Moteur de contrôle Rust : course témoin (2 variantes, graine 7) + contre-vérification croisée ===
{
    use ndarray::Array2;
    use rand::{rngs::StdRng, Rng, SeedableRng};
    use rand_distr::{Distribution, Normal};
    use std::time::Instant;

    const PUZZLE_LINE_28: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const POPULATION: usize = 50;
    const GENERATIONS: usize = 160;
    const REFERENCE_EVALS: usize = 8_050;
    const REFERENCE_CONFLICTS: usize = 26;
    // La suite StdRng n'est pas celle de NumPy ; seuls les invariants sont comparés à l'égalité.
    const CONFLICT_TOLERANCE: usize = 12;

    #[derive(Debug)]
    enum SolveError { InvalidConfig, InvalidPuzzle, InvalidGenes, InvalidNormal, EmptyPopulation, Json, Control }

    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Port { Aligned, Literal }

    #[derive(Clone)]
    struct Candidate { genes: Vec<f64>, conflicts: usize }
    struct Particle { current: Candidate, pbest: Candidate }
    struct Genome { best: Candidate, evals: usize, checkpoints: [usize; 4] }

    fn puzzle_28() -> Result<[u8; 81], SolveError> {
        let mut grid = [0; 81];
        if PUZZLE_LINE_28.len() != grid.len() { return Err(SolveError::InvalidPuzzle); }
        for (slot, digit) in grid.iter_mut().zip(PUZZLE_LINE_28.bytes()) {
            if !digit.is_ascii_digit() { return Err(SolveError::InvalidPuzzle); }
            *slot = digit - b'0';
        }
        Ok(grid)
    }

    // La voie de contrôle compte les occurrences par unité, sans utiliser les masques du noyau.
    fn evaluate(puzzle: &[u8; 81], empty: &[usize], genes: Vec<f64>, evals: &mut usize) -> Result<Candidate, SolveError> {
        let mut grid = *puzzle;
        let used = genes.get(..empty.len()).ok_or(SolveError::InvalidGenes)?;
        for (&i, &gene) in empty.iter().zip(used) {
            if !gene.is_finite() { return Err(SolveError::InvalidGenes); }
            let bounded = gene.clamp(1.0, 9.0);
            let low = bounded.floor() as u8;
            let fraction = bounded - f64::from(low);
            grid[i] = low + u8::from(fraction > 0.5 || (fraction == 0.5 && low % 2 == 1));
        }
        let mut counts = [[0_u8; 10]; 27];
        for (i, &digit) in grid.iter().enumerate() {
            let (r, c, d) = (i / 9, i % 9, usize::from(digit));
            counts[r][d] += 1;
            counts[9 + c][d] += 1;
            counts[18 + 3 * (r / 3) + c / 3][d] += 1;
        }
        let conflicts = counts.iter().flatten().map(|&n| usize::from(n.saturating_sub(1))).sum();
        *evals += 1;
        Ok(Candidate { genes, conflicts })
    }

    fn sample_normal(rng: &mut StdRng, mean: f64, std: f64) -> Result<f64, SolveError> {
        if std == 0.0 { return Ok(mean); }
        Ok(Normal::new(mean, std).map_err(|_| SolveError::InvalidNormal)?.sample(rng))
    }

    fn run_port(port: Port, seed: u64, pop_size: usize, generations: usize, std_floor: f64) -> Result<Genome, SolveError> {
        if pop_size < 2 || generations < 4 || generations.checked_add(1).and_then(|n| n.checked_mul(pop_size)).is_none()
            || !std_floor.is_finite() || std_floor < 0.0 {
            return Err(SolveError::InvalidConfig);
        }
        let puzzle = puzzle_28()?;
        let empty: Vec<_> = puzzle.iter().enumerate().filter_map(|(i, &d)| (d == 0).then_some(i)).collect();
        // La population initiale est créée et évaluée après le seeding, avant la première génération.
        let mut rng = StdRng::seed_from_u64(seed);
        let mut evals = 0;
        let mut population = Vec::with_capacity(pop_size);
        for _ in 0..pop_size {
            let genes = (0..empty.len()).map(|_| rng.random_range(1.0..10.0)).collect();
            let current = evaluate(&puzzle, &empty, genes, &mut evals)?;
            population.push(Particle { pbest: current.clone(), current });
        }
        let mut best = population.iter().map(|p| &p.pbest).min_by_key(|p| p.conflicts)
            .ok_or(SolveError::EmptyPopulation)?.clone();
        let quarters = [generations / 4, generations / 2, 3 * (generations / 4) + 3 * (generations % 4) / 4, generations];
        let mut checkpoints = [0; 4];
        for generation in 1..=generations {
            // Même gbest pour toute la génération et mêmes départages stricts dans les deux ports.
            for particle in &mut population {
                let anchor = if port == Port::Literal { &particle.pbest } else { &particle.current };
                let mut genes = Vec::with_capacity(empty.len());
                for (&a, &g) in anchor.genes.iter().zip(&best.genes) {
                    let sampled = sample_normal(&mut rng, (a + g) / 2.0, (g - a).abs().max(std_floor))?;
                    genes.push(sampled.clamp(1.0, 10.0));
                }
                let child = evaluate(&puzzle, &empty, genes, &mut evals)?;
                if child.conflicts < particle.pbest.conflicts { particle.pbest = child.clone(); }
                if port == Port::Literal || child.conflicts < particle.current.conflicts { particle.current = child; }
            }
            for particle in &population {
                if particle.pbest.conflicts < best.conflicts { best = particle.pbest.clone(); }
            }
            if let Some(q) = quarters.iter().position(|&mark| mark == generation) { checkpoints[q] = best.conflicts; }
        }
        Ok(Genome { best, evals, checkpoints })
    }

    // La voie du socle redécode les gènes en ndarray, puis compte les doublons par masques.
    fn recost(puzzle: &[u8; 81], genes: &[f64]) -> Result<usize, SolveError> {
        let digits = puzzle.iter().copied().map(i64::from).collect();
        let mut grid = Array2::from_shape_vec((9, 9), digits).map_err(|_| SolveError::InvalidPuzzle)?;
        let mut genes = genes.iter().copied();
        for digit in grid.iter_mut() {
            if *digit == 0 {
                let gene = genes.next().ok_or(SolveError::InvalidGenes)?;
                if !gene.is_finite() { return Err(SolveError::InvalidGenes); }
                *digit = gene.round_ties_even().clamp(1.0, 9.0) as i64;
            }
        }
        let mut cost = 0;
        for i in 0..9 {
            let mut seen = [0_u16; 3];
            for j in 0..9 {
                let digits = [grid[[i, j]], grid[[j, i]], grid[[3 * (i / 3) + j / 3, 3 * (i % 3) + j % 3]]];
                for (mask, digit) in seen.iter_mut().zip(digits) {
                    let bit = 1_u16 << (digit as u32);
                    cost += ((*mask & bit) != 0) as usize;
                    *mask |= bit;
                }
            }
        }
        Ok(cost)
    }

    let result = (|| -> Result<(), SolveError> {
        // Échauffement symétrique (course jetée), puis courses témoins graine 7.
        let _warmup = run_port(Port::Aligned, 123, POPULATION, 10, 0.0)?;
        let puzzle = puzzle_28()?;
        let mut conforming = true;
        for (port, label) in [(Port::Literal, "port-littéral"), (Port::Aligned, "port-aligné")] {
            let start = Instant::now();
            let result = run_port(port, 7, POPULATION, GENERATIONS, 0.0)?;
            let ms = start.elapsed().as_secs_f64() * 1_000.0;
            println!("Rust BBPSO {label} (graine 7, témoin) : {} conflits, {} évaluations, {ms:.0} ms.", result.best.conflicts, result.evals);
            let budget_ok = result.evals == REFERENCE_EVALS && result.evals == POPULATION * (GENERATIONS + 1);
            println!("Budget : calculé {} | attendu {REFERENCE_EVALS} -> {}", result.evals, if budget_ok { "CONFORME" } else { "ÉCART" });
            let delta = result.best.conflicts.abs_diff(REFERENCE_CONFLICTS);
            let band_ok = delta <= CONFLICT_TOLERANCE;
            println!("Score : calculé {} | référence mealpy {REFERENCE_CONFLICTS} | écart absolu {delta} (tolérance {CONFLICT_TOLERANCE}) -> {}", result.best.conflicts, if band_ok { "CONFORME" } else { "ÉCART" });

            // Contre-vérification croisée : le vainqueur du port, décodé et costé par la voie du socle.
            let json = serde_json::to_string(&result.best.genes).map_err(|_| SolveError::Json)?;
            let genes: Vec<f64> = serde_json::from_str(&json).map_err(|_| SolveError::Json)?;
            let cost = recost(&puzzle, &genes)?;
            let equal = cost == result.best.conflicts;
            println!("Contre-vérif croisée : coût du meilleur {label} = {cost} (le port rapporte {}) -> {}", result.best.conflicts, if equal { "IDENTIQUE" } else { "ÉCART" });
            conforming &= budget_ok && band_ok && equal && genes == result.best.genes
                && result.checkpoints[3] == result.best.conflicts && result.checkpoints.windows(2).all(|p| p[0] >= p[1]);
        }
        if conforming { Ok(()) } else { Err(SolveError::Control) }
    })();
    if let Err(error) = result { eprintln!("Échec du contrôle des courses témoins : {error:?}"); }
};

Rust BBPSO port-littéral (graine 7, témoin) : 25 conflits, 8050 évaluations, 14 ms.


Budget : calculé 8050 | attendu 8050 -> CONFORME


Score : calculé 25 | référence mealpy 26 | écart absolu 1 (tolérance 12) -> CONFORME


Contre-vérif croisée : coût du meilleur port-littéral = 25 (le port rapporte 25) -> IDENTIQUE


Rust BBPSO port-aligné (graine 7, témoin) : 25 conflits, 8050 évaluations, 13 ms.


Budget : calculé 8050 | attendu 8050 -> CONFORME


Score : calculé 25 | référence mealpy 26 | écart absolu 1 (tolérance 12) -> CONFORME


Contre-vérif croisée : coût du meilleur port-aligné = 25 (le port rapporte 25) -> IDENTIQUE


***

## Le croisement : trois bras et quatre graines appariées

Chaque couple bras–graine est rejoué trois fois. Le noyau compte l'initialisation comme première génération, soit 8 000 évaluations ; les deux ports l'évaluent avant les 160 générations, soit 8 050 évaluations. L'appariement porte sur la grille, les graines, la taille de population et le nombre de générations, pas sur une égalité stricte du nombre d'évaluations.

In [7]:
// === LE BENCH : 3 bras x 4 graines {0,1,7,42}, population 50, 160 générations ===
// Bras 1 noyau BBPSO | Bras 2 port-aligné (même ancre) | Bras 3 port-littéral (Kennedy 2003).
{
    use ndarray::Array2;
    use rand::{rngs::StdRng, Rng, SeedableRng};
    use rand_distr::{Distribution, Normal};
    use std::time::Instant;

    const PUZZLE_LINE_28: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const SEEDS: [u64; 4] = [0, 1, 7, 42];
    const POPULATION: usize = 50;
    const GENERATIONS: usize = 160;
    const REPEATS: usize = 3;
    const NAMES: [&str; 3] = ["noyau", "port-aligné", "port-littéral"];
    // Références .NET/mealpy du tableau d'origine, dans le même ordre de bras et de graines.
    const REF_EVALS: [usize; 3] = [8_000, 8_050, 8_050];
    const REF_SCORES: [[usize; 4]; 3] = [[31, 31, 29, 30], [19, 25, 26, 27], [19, 25, 26, 27]];
    const REF_KERNEL_CP: [[usize; 4]; 4] = [[33, 31, 31, 31], [31; 4], [29; 4], [30; 4]];
    const REF_ANCHOR_DELTA: [i64; 4] = [0; 4];
    const REF_DETERMINISM: usize = 12;
    // Comparaison de scores sous bande, car StdRng ne recrée ni les tirages .NET ni ceux de NumPy.
    const CONFLICT_TOLERANCE: usize = 12;

    #[derive(Debug)]
    enum SolveError { InvalidConfig, InvalidPuzzle, InvalidGenes, InvalidNormal, EmptyPopulation, Control }
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Arm { Kernel, Aligned, Literal }
    #[derive(Clone, PartialEq)]
    struct Candidate { genes: Vec<f64>, conflicts: usize }
    struct Particle { current: Candidate, pbest: Candidate }
    #[derive(PartialEq)]
    struct Genome { best: Candidate, evals: usize, checkpoints: [usize; 4] }
    struct BenchRow { run: Genome, median_ms: f64, all_same: bool }

    fn parse_puzzle_28() -> Result<[u8; 81], SolveError> {
        let mut puzzle = [0; 81];
        if PUZZLE_LINE_28.len() != puzzle.len() { return Err(SolveError::InvalidPuzzle); }
        for (slot, byte) in puzzle.iter_mut().zip(PUZZLE_LINE_28.bytes()) {
            if !byte.is_ascii_digit() { return Err(SolveError::InvalidPuzzle); }
            *slot = byte - b'0';
        }
        Ok(puzzle)
    }

    fn eval_kernel(puzzle: &Array2<i64>, empty: &[usize], genes: Vec<f64>, evals: &mut usize) -> Result<Candidate, SolveError> {
        let mut grid = puzzle.clone();
        for (&i, &gene) in empty.iter().zip(genes.get(..empty.len()).ok_or(SolveError::InvalidGenes)?) {
            if !gene.is_finite() { return Err(SolveError::InvalidGenes); }
            grid[[i / 9, i % 9]] = gene.round_ties_even().clamp(1.0, 9.0) as i64;
        }
        let mut conflicts = 0;
        for i in 0..9 {
            let mut seen = [0_u16; 3];
            for j in 0..9 {
                let digits = [grid[[i, j]], grid[[j, i]], grid[[3 * (i / 3) + j / 3, 3 * (i % 3) + j % 3]]];
                for (mask, digit) in seen.iter_mut().zip(digits) {
                    let bit = 1_u16 << (digit as u32);
                    conflicts += ((*mask & bit) != 0) as usize;
                    *mask |= bit;
                }
            }
        }
        *evals += 1;
        Ok(Candidate { genes, conflicts })
    }

    fn eval_port(puzzle: &[u8; 81], empty: &[usize], genes: Vec<f64>, evals: &mut usize) -> Result<Candidate, SolveError> {
        let mut grid = *puzzle;
        for (&i, &gene) in empty.iter().zip(genes.get(..empty.len()).ok_or(SolveError::InvalidGenes)?) {
            if !gene.is_finite() { return Err(SolveError::InvalidGenes); }
            let bounded = gene.clamp(1.0, 9.0);
            let low = bounded.floor() as u8;
            let fraction = bounded - f64::from(low);
            grid[i] = low + u8::from(fraction > 0.5 || (fraction == 0.5 && low % 2 == 1));
        }
        let mut counts = [[0_u8; 10]; 27];
        for (i, &digit) in grid.iter().enumerate() {
            let (r, c, d) = (i / 9, i % 9, usize::from(digit));
            counts[r][d] += 1;
            counts[9 + c][d] += 1;
            counts[18 + 3 * (r / 3) + c / 3][d] += 1;
        }
        let conflicts = counts.iter().flatten().map(|&n| usize::from(n.saturating_sub(1))).sum();
        *evals += 1;
        Ok(Candidate { genes, conflicts })
    }

    fn sample_normal(rng: &mut StdRng, mean: f64, std: f64) -> Result<f64, SolveError> {
        if std == 0.0 { return Ok(mean); }
        Ok(Normal::new(mean, std).map_err(|_| SolveError::InvalidNormal)?.sample(rng))
    }

    fn propose(rng: &mut StdRng, anchor: &[f64], global: &[f64], std_floor: f64) -> Result<Vec<f64>, SolveError> {
        anchor.iter().zip(global).map(|(&a, &g)| {
            let x = sample_normal(rng, (a + g) / 2.0, (g - a).abs().max(std_floor))?;
            Ok(x.clamp(1.0, 10.0))
        }).collect()
    }

    fn solve(arm: Arm, seed: u64, pop_size: usize, generations: usize, std_floor: f64) -> Result<Genome, SolveError> {
        if pop_size < 2 || generations < 4 || generations.checked_add(1).and_then(|n| n.checked_mul(pop_size)).is_none()
            || !std_floor.is_finite() || std_floor < 0.0 {
            return Err(SolveError::InvalidConfig);
        }
        let puzzle = parse_puzzle_28()?;
        let empty: Vec<_> = puzzle.iter().enumerate().filter_map(|(i, &v)| (v == 0).then_some(i)).collect();
        let quarters = [generations / 4, generations / 2, 3 * (generations / 4) + 3 * (generations % 4) / 4, generations];
        let mut checkpoints = [0; 4];
        let mut evals = 0;
        // Graine neuve avant la création de la population à chaque répétition.
        let mut rng = StdRng::seed_from_u64(seed);

        // --- Bras 1 : noyau, 3 répétitions par graine, ms = médiane ---
        if arm == Arm::Kernel {
            let digits = puzzle.iter().copied().map(i64::from).collect();
            let grid = Array2::from_shape_vec((9, 9), digits).map_err(|_| SolveError::InvalidPuzzle)?;
            let mut pop = Vec::with_capacity(2 * pop_size);
            for _ in 0..pop_size {
                let genes = (0..empty.len()).map(|_| rng.random_range(1.0..10.0)).collect();
                pop.push(eval_kernel(&grid, &empty, genes, &mut evals)?);
            }
            pop.sort_by_key(|p| p.conflicts);
            // Comme dans le témoin du noyau, l'initialisation tient lieu de génération 1.
            for generation in 1..=generations {
                if generation > 1 {
                    let best = &pop.first().ok_or(SolveError::EmptyPopulation)?.genes;
                    let mut children = Vec::with_capacity(pop_size);
                    for parent in &pop {
                        let genes = propose(&mut rng, &parent.genes, best, std_floor)?;
                        children.push(eval_kernel(&grid, &empty, genes, &mut evals)?);
                    }
                    pop.extend(children);
                    // Tri stable : à égalité, un parent passe avant un enfant de cette génération.
                    pop.sort_by_key(|p| p.conflicts);
                    pop.truncate(pop_size);
                }
                if let Some(q) = quarters.iter().position(|&mark| mark == generation) {
                    checkpoints[q] = pop.first().ok_or(SolveError::EmptyPopulation)?.conflicts;
                }
            }
            let best = pop.into_iter().next().ok_or(SolveError::EmptyPopulation)?;
            return Ok(Genome { best, evals, checkpoints });
        }

        // --- Bras 2 et 3 : harnais de contrôle Rust, boucle commune aux deux ports ---
        let mut pop = Vec::with_capacity(pop_size);
        for _ in 0..pop_size {
            let genes = (0..empty.len()).map(|_| rng.random_range(1.0..10.0)).collect();
            let current = eval_port(&puzzle, &empty, genes, &mut evals)?;
            pop.push(Particle { pbest: current.clone(), current });
        }
        let mut best = pop.iter().map(|p| &p.pbest).min_by_key(|p| p.conflicts).ok_or(SolveError::EmptyPopulation)?.clone();
        for generation in 1..=generations {
            for particle in &mut pop {
                let anchor = if arm == Arm::Literal { &particle.pbest } else { &particle.current };
                let genes = propose(&mut rng, &anchor.genes, &best.genes, std_floor)?;
                let child = eval_port(&puzzle, &empty, genes, &mut evals)?;
                // Les égalités ne changent pas pbest, ni la position dans le port-aligné.
                if child.conflicts < particle.pbest.conflicts { particle.pbest = child.clone(); }
                if arm == Arm::Literal || child.conflicts < particle.current.conflicts { particle.current = child; }
            }
            for particle in &pop {
                if particle.pbest.conflicts < best.conflicts { best = particle.pbest.clone(); }
            }
            if let Some(q) = quarters.iter().position(|&mark| mark == generation) { checkpoints[q] = best.conflicts; }
        }
        Ok(Genome { best, evals, checkpoints })
    }

    fn bench(arm: Arm, seed: u64) -> Result<BenchRow, SolveError> {
        let mut runs = Vec::with_capacity(REPEATS);
        let mut times = [0.0_f64; REPEATS];
        for time in &mut times {
            let start = Instant::now();
            let run = solve(arm, seed, POPULATION, GENERATIONS, 0.0)?;
            *time = start.elapsed().as_secs_f64() * 1_000.0;
            runs.push(run);
        }
        times.sort_by(f64::total_cmp);
        let mut runs = runs.into_iter();
        let run = runs.next().ok_or(SolveError::EmptyPopulation)?;
        // Comparer aussi les checkpoints, le budget et l'incumbent ; jamais les temps.
        let all_same = runs.all(|other| other == run);
        Ok(BenchRow { run, median_ms: times[REPEATS / 2], all_same })
    }

    fn summary(mut scores: [usize; 4]) -> (f64, usize, usize) {
        scores.sort_unstable();
        ((scores[1] as f64 + scores[2] as f64) / 2.0, scores[0], scores[3])
    }

    fn shape_of(delta: [i64; 4]) -> &'static str {
        let s = delta.map(i64::signum);
        if s == [0; 4] { return "nul (identiques)"; }
        let early_same = s[0] != 0 && s[0] == s[1];
        if early_same && s.iter().all(|&v| v == s[0]) {
            return if s[0] < 0 { "persistant (bras A devant)" } else { "persistant (bras B devant)" };
        }
        if early_same && s[3] != 0 && s[3] != s[0] { return "croisement"; }
        if s[3] == 0 { return "refermeture"; }
        "mixte"
    }

    let result = (|| -> Result<(), SolveError> {
        let mut rows: [Vec<BenchRow>; 3] = std::array::from_fn(|_| Vec::with_capacity(SEEDS.len()));
        for (k, arm) in [Arm::Kernel, Arm::Aligned, Arm::Literal].into_iter().enumerate() {
            for seed in SEEDS { rows[k].push(bench(arm, seed)?); }
        }

        // --- Table : conflits, budget mesuré, coût par éval, checkpoints 25/50/75/100 % ---
        println!("{:<16} {:>6} {:>9} {:>7} {:>7} {:>8} {:<18}", "moteur", "graine", "conflits", "evals", "ms", "ms/eval", "cp25/50/75/100");
        for (a, arm_rows) in rows.iter().enumerate() {
            for (&seed, row) in SEEDS.iter().zip(arm_rows) {
                let cp = row.run.checkpoints.map(|v| v.to_string()).join("/");
                println!("{:<16} {:>6} {:>9} {:>7} {:>7.0} {:>8.3} {:<18}", NAMES[a], seed,
                    row.run.best.conflicts, row.run.evals, row.median_ms, row.median_ms / row.run.evals as f64, cp);
            }
        }
        let stats = std::array::from_fn::<_, 3, _>(|a| summary(std::array::from_fn(|s| rows[a][s].run.best.conflicts)));
        let avg_ms = rows.each_ref().map(|arm| arm.iter().map(|row| row.median_ms / row.run.evals as f64).sum::<f64>() / SEEDS.len() as f64);
        println!();
        for (a, &(m, lo, hi)) in stats.iter().enumerate() {
            println!("{:<16} : médiane conflits {m:.1} (min {lo}, max {hi}), ms/éval moyen {:.3}", NAMES[a], avg_ms[a]);
        }
        println!();
        for (label, a, b) in [("NOYAU", 0, 1), ("D'ANCRAGE", 1, 2), ("BRUT", 0, 2)] {
            let (ma, mb) = (stats[a].0, stats[b].0);
            let final_delta = std::array::from_fn::<_, 4, _>(|s| rows[a][s].run.best.conflicts as i64 - rows[b][s].run.best.conflicts as i64);
            if a == 0 && avg_ms[b] > 0.0 {
                println!("ÉCART {label} : médianes {ma:.1} vs {mb:.1}, delta {:+.1}, par graine {final_delta:?}, ms/éval {:.2}x", ma - mb, avg_ms[a] / avg_ms[b]);
            } else {
                println!("ÉCART {label} : médianes {ma:.1} vs {mb:.1}, delta {:+.1}, par graine {final_delta:?}", ma - mb);
            }
        }

        // --- Classement dynamique : rangs dérivés des mesures, avec ex æquo ---
        let mut rank = [0, 1, 2];
        rank.sort_by(|&a, &b| stats[a].0.total_cmp(&stats[b].0));
        let mut ranking = String::new();
        for (i, &a) in rank.iter().enumerate() {
            if i > 0 { ranking.push_str(if stats[rank[i - 1]].0 == stats[a].0 { " = " } else { " < " }); }
            ranking.push_str(&format!("{} ({:.1})", NAMES[a], stats[a].0));
        }
        println!("\nClassement qualité (médiane conflits croissante) : {ranking}");

        // --- Forme de l'écart aux checkpoints : noyau - aligné et aligné - littéral ---
        // delta < 0 => le premier bras est DEVANT (moins de conflits) à ce checkpoint.
        println!("\nForme noyau et ancrage, par graine :");
        let mut anchor_ok = true;
        for (s, seed) in SEEDS.into_iter().enumerate() {
            let [k, a, l] = rows.each_ref().map(|arm| arm[s].run.checkpoints);
            let dn = std::array::from_fn(|q| k[q] as i64 - a[q] as i64);
            let da = std::array::from_fn(|q| a[q] as i64 - l[q] as i64);
            let ok = da == REF_ANCHOR_DELTA && rows[1][s].run.best.conflicts == rows[2][s].run.best.conflicts;
            anchor_ok &= ok;
            println!("  graine {seed:>2} : noyau {dn:?} {:<28} | ancrage {da:?} {} | attendu {REF_ANCHOR_DELTA:?} -> {}", shape_of(dn), shape_of(da), if ok { "CONFORME" } else { "ÉCART" });
        }
        let det = rows.iter().flatten().filter(|r| r.all_same).count();
        println!("\nDéterminisme (3 répétitions identiques par graine) : {det}/12 bras-graines stables (attendu {REF_DETERMINISM}/12) -> {}", if det == REF_DETERMINISM { "CONFORME" } else { "ÉCART" });

        // Conformité différentielle : comparer chaque cas, sans laisser les durées décider du verdict.
        let mut all_ok = anchor_ok && det == REF_DETERMINISM;
        println!("\nÉcarts absolus par rapport au carnet .NET/mealpy (scores : tolérance {CONFLICT_TOLERANCE} ; budgets : 0) :");
        for (a, arm_rows) in rows.iter().enumerate() {
            for (s, row) in arm_rows.iter().enumerate() {
                let score = row.run.best.conflicts;
                let delta = score.abs_diff(REF_SCORES[a][s]);
                let eval_delta = row.run.evals.abs_diff(REF_EVALS[a]);
                let cp = row.run.checkpoints;
                let trajectory_ok = cp[3] == score && cp.windows(2).all(|p| p[0] >= p[1]);
                let ok = delta <= CONFLICT_TOLERANCE && eval_delta == 0 && trajectory_ok;
                all_ok &= ok;
                println!("  {} graine {} : score {score}/{} (écart {delta}), budget {}/{} (écart {eval_delta}), trajectoire {} -> {}", NAMES[a], SEEDS[s], REF_SCORES[a][s], row.run.evals, REF_EVALS[a], if trajectory_ok { "cohérente" } else { "incohérente" }, if ok { "CONFORME" } else { "ÉCART" });
                if a == 0 {
                    let reference = REF_KERNEL_CP[s];
                    let distances = std::array::from_fn::<_, 4, _>(|q| cp[q].abs_diff(reference[q]));
                    let ok = distances.iter().all(|&d| d <= CONFLICT_TOLERANCE);
                    all_ok &= ok;
                    println!("    checkpoints noyau : calculé {cp:?} | référence {reference:?} | écarts {distances:?} -> {}", if ok { "CONFORME" } else { "ÉCART" });
                }
            }
            let (m, lo, hi) = stats[a];
            let (rm, rlo, rhi) = summary(REF_SCORES[a]);
            let (dm, dlo, dhi) = ((m - rm).abs(), lo.abs_diff(rlo), hi.abs_diff(rhi));
            let ok = dm <= CONFLICT_TOLERANCE as f64 && dlo <= CONFLICT_TOLERANCE && dhi <= CONFLICT_TOLERANCE;
            all_ok &= ok;
            println!("  {} : médiane {m:.1}/{rm:.1}, min {lo}/{rlo}, max {hi}/{rhi} ; écarts {dm:.1}/{dlo}/{dhi} -> {}", NAMES[a], if ok { "CONFORME" } else { "ÉCART" });
        }
        if all_ok { Ok(()) } else { Err(SolveError::Control) }
    })();
    if let Err(error) = result { eprintln!("Échec d'un contrôle du banc : {error:?}"); }
};

moteur           graine  conflits   evals      ms  ms/eval cp25/50/75/100    


noyau                 0        36    8000      18    0.002 36/36/36/36       


noyau                 1        37    8000      20    0.003 39/37/37/37       


noyau                 7        24    8000      20    0.002 24/24/24/24       


noyau                42        34    8000      19    0.002 34/34/34/34       


port-aligné           0        26    8050      15    0.002 38/28/26/26       


... (36 outputs left out)

  port-littéral graine 1 : score 25/25 (écart 0), budget 8050/8050 (écart 0), trajectoire cohérente -> CONFORME


  port-littéral graine 7 : score 25/26 (écart 1), budget 8050/8050 (écart 0), trajectoire cohérente -> CONFORME


  port-littéral graine 42 : score 23/27 (écart 4), budget 8050/8050 (écart 0), trajectoire cohérente -> CONFORME


  port-littéral : médiane 25.0/25.5, min 23/19, max 26/27 ; écarts 0.5/4/1 -> CONFORME


In [8]:
// === Coût par évaluation : la fitness seule, hors moteur, 500 vecteurs identiques ===
// Les vecteurs (LCG, graines 42..541), passés en JSON au Python dans le carnet d'origine,
// sont ici passés en JSON à la seconde voie Rust : les DEUX voies chronomètrent le même travail
// de décodage + coût sur les mêmes 500 points, sans inclure le transfert dans le chronométrage.
{
    use ndarray::Array2;
    use std::{hint::black_box, time::Instant};

    const PUZZLE_LINE_28: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    const K28: usize = 500;
    const REPEATS: usize = 5;

    fn lcg_vector_28(mut state: u32, n: usize) -> Vec<f64> {
        (0..n).map(|_| {
            state = state.wrapping_mul(1_664_525).wrapping_add(1_013_904_223);
            1.0 + (f64::from(state) / 4_294_967_296.0) * 9.0
        }).collect()
    }

    fn decode_a(genes: &[f64]) -> Result<Array2<i64>, &'static str> {
        if PUZZLE_LINE_28.len() != 81 || !PUZZLE_LINE_28.bytes().all(|b| b.is_ascii_digit()) {
            return Err("grille de référence invalide");
        }
        let digits = PUZZLE_LINE_28.bytes().map(|b| i64::from(b - b'0')).collect();
        let mut grid = Array2::from_shape_vec((9, 9), digits).map_err(|_| "forme de grille invalide")?;
        let mut genes = genes.iter().copied();
        for digit in grid.iter_mut() {
            if *digit == 0 {
                let gene = genes.next().ok_or("vecteur R1 trop court")?;
                if !gene.is_finite() { return Err("gène non fini"); }
                *digit = gene.round_ties_even().clamp(1.0, 9.0) as i64;
            }
        }
        Ok(grid)
    }

    // Voie A : masques de présence, et un conflit pour chaque occurrence après la première.
    fn cost_a(grid: &Array2<i64>) -> usize {
        let mut cost = 0;
        for i in 0..9 {
            let mut seen = [0_u16; 3];
            for j in 0..9 {
                let digits = [grid[[i, j]], grid[[j, i]], grid[[3 * (i / 3) + j / 3, 3 * (i % 3) + j % 3]]];
                for (mask, digit) in seen.iter_mut().zip(digits) {
                    let bit = 1_u16 << (digit as u32);
                    cost += ((*mask & bit) != 0) as usize;
                    *mask |= bit;
                }
            }
        }
        cost
    }

    fn decode_b(genes: &[f64]) -> Result<[u8; 81], &'static str> {
        let bytes = PUZZLE_LINE_28.as_bytes();
        if bytes.len() != 81 { return Err("longueur de grille invalide"); }
        let mut grid = [0; 81];
        let mut k = 0;
        for (slot, &byte) in grid.iter_mut().zip(bytes) {
            *slot = match byte {
                b'1'..=b'9' => byte - b'0',
                b'0' => {
                    let &gene = genes.get(k).ok_or("vecteur R1 trop court")?;
                    k += 1;
                    if !gene.is_finite() { return Err("gène non fini"); }
                    let bounded = gene.clamp(1.0, 9.0);
                    let low = bounded.floor() as u8;
                    let fraction = bounded - f64::from(low);
                    low + u8::from(fraction > 0.5 || (fraction == 0.5 && low % 2 == 1))
                }
                _ => return Err("indice de grille invalide"),
            };
        }
        Ok(grid)
    }

    // Voie B : histogrammes indépendants, avec contribution max(effectif - 1, 0).
    fn cost_b(grid: &[u8; 81]) -> usize {
        let mut counts = [[0_u8; 10]; 27];
        for (i, &digit) in grid.iter().enumerate() {
            let (r, c, d) = (i / 9, i % 9, usize::from(digit));
            counts[r][d] += 1;
            counts[9 + c][d] += 1;
            counts[18 + 3 * (r / 3) + c / 3][d] += 1;
        }
        counts.iter().flatten().map(|&n| usize::from(n.saturating_sub(1))).sum()
    }

    fn time_costs(vectors: &[Vec<f64>], evaluate: impl Fn(&[f64]) -> Result<usize, &'static str>) -> Result<f64, &'static str> {
        let mut times = [0.0_f64; REPEATS];
        for time in &mut times {
            let start = Instant::now();
            for vector in vectors { black_box(evaluate(black_box(vector.as_slice()))?); }
            *time = start.elapsed().as_secs_f64() * 1_000.0;
        }
        times.sort_by(f64::total_cmp);
        Ok(times[REPEATS / 2])
    }

    let result = (|| -> Result<(), String> {
        let vectors: Vec<_> = (42..=541).map(|seed| lcg_vector_28(seed, 51)).collect();
        let json = serde_json::to_string(&vectors).map_err(|e| e.to_string())?;
        let received: Vec<Vec<f64>> = serde_json::from_str(&json).map_err(|e| e.to_string())?;
        if vectors.len() != K28 || vectors != received { return Err("régénération ou transfert des vecteurs incorrect".into()); }

        // Vérifier les coûts point par point hors chronomètre, et non seulement leur somme.
        let mut agreement = 0;
        let mut max_delta = 0;
        for (left, right) in vectors.iter().zip(&received) {
            let a = cost_a(&decode_a(left)?);
            let b = cost_b(&decode_b(right)?);
            agreement += (a == b) as usize;
            max_delta = max_delta.max(a.abs_diff(b));
        }
        let ok = agreement == K28 && max_delta == 0;
        println!("Accord des coûts : {agreement}/{K28} (attendu {K28}/{K28}), écart maximal {max_delta} (attendu 0) -> {}", if ok { "IDENTIQUE" } else { "ÉCART" });
        if !ok { return Err("les coûts divergent sur les vecteurs chronométrés".into()); }

        let a_ms = time_costs(&vectors, |v| Ok(cost_a(&decode_a(v)?)))?;
        let b_ms = time_costs(&received, |v| Ok(cost_b(&decode_b(v)?)))?;
        println!("Fitness seule, {K28} vecteurs identiques (médiane de {REPEATS} répétitions par côté) :");
        println!("  Rust A : {a_ms:.1} ms total -> {:.3} ms/éval", a_ms / K28 as f64);
        println!("  Rust B : {b_ms:.1} ms total -> {:.3} ms/éval", b_ms / K28 as f64);
        if a_ms <= 0.0 { return Err("durée de la voie A sous la résolution du chronomètre".into()); }
        println!("  rapport Rust B/Rust A : {:.2}x", b_ms / a_ms);
        Ok(())
    })();
    if let Err(error) = result { eprintln!("Échec de la mesure de fitness : {error}"); }
};

Accord des coûts : 500/500 (attendu 500/500), écart maximal 0 (attendu 0) -> IDENTIQUE


Fitness seule, 500 vecteurs identiques (médiane de 5 répétitions par côté) :


  Rust A : 1.1 ms total -> 0.002 ms/éval


  Rust B : 0.7 ms total -> 0.001 ms/éval


  rapport Rust B/Rust A : 0.65x


**Lecture du croisement.**

- **Classement** : les médianes de conflits, recalculées sur les quatre graines, donnent le classement Rust ci-dessus. Dans le carnet d'origine, les deux jumeaux mealpy étaient ex æquo à 25,5 conflits chacun, devant MGS (30,5). Ces valeurs sont des références de comparaison, pas des résultats Rust : à graine égale, `StdRng` ne donne pas les mêmes tirages.
- **Écart d'ancrage** : lisez les différences aux quatre checkpoints pour chaque graine. Dans le port-aligné, l'amélioration stricte fait coïncider la position courante et le meilleur personnel. Dans le port-littéral, un remplacement peut dégrader la position courante, mais le tirage suivant reste ancré sur le meilleur personnel. Les deux ports peuvent donc suivre les mêmes meilleurs coûts sans suivre les mêmes positions courantes. Le banc contrôle explicitement l'écart nul sur les scores finaux et les checkpoints.
- **Écart noyau** : les profils graine par graine distinguent un écart persistant, un croisement, une refermeture et un profil mixte. Un delta négatif signifie que le noyau est devant à ce checkpoint ; l'égalité `cp25 = cp100` signale une absence d'amélioration du meilleur coût après le premier quart, pas l'immobilité de toute la population. Le noyau et le port-aligné conservent la même ancre mais ne retiennent pas les enfants de la même façon et ne comptent pas l'initialisation de la même manière : leur écart ne mesure pas un effet isolé du moteur.
- **Vitesse** : le `ms/éval` du banc inclut le travail du moteur ; le rapport de la fitness isolée compare les deux voies Rust sur les mêmes vecteurs. Ce sont des mesures **machine** : les rapports eux-mêmes varient d'un passage du noyau à l'autre et le verdict de conformité ne repose pas sur eux. Le budget apparié est de 8 000 évaluations pour le noyau et de 8 050 pour les ports.

## Exercice 1 : budget ×4 — l'écart se referme-t-il ou s'installe-t-il ?

Le croisement s'est joué à budget 8 000 évaluations pour le noyau et 8 050 pour les ports. Refaites-le à budget ×4 (population 50,
640 générations, 4 graines, les trois bras) et comparez : l'écart de médiane se
referme, se stabilise, ou s'agrandit ? Relevez aussi les checkpoints 25/50/75/100 % — la
*forme* de l'écart à budget long est-elle la même qu'à court ?

**Indice** : reprenez la cellule autonome du banc ; `solve` renvoie déjà les checkpoints pour chacun des trois bras. Gardez le décompte de l'initialisation propre à chaque bras.

In [9]:
// Exercice 1 — budget ×4 : 50 particules, 640 générations, quatre graines, trois bras.
// Dans une copie de la cellule 12, insérez ce bloc dans la fermeture `result`,
// juste avant `Ok(())`. Laissez le banc à 160 générations pour servir de référence.
// for (arm, label) in [(Arm::Kernel, "noyau"), (Arm::Aligned, "port-aligné"), (Arm::Literal, "port-littéral")] {
//     for seed in SEEDS {
//         let run = solve(arm, seed, POPULATION, 640, 0.0)?;
//         println!("{label}, graine {seed} : {} conflits, {} évaluations, checkpoints {:?}",
//             run.best.conflicts, run.evals, run.checkpoints);
//     }
// }
// À compléter : calculez les écarts de médiane et comparez les profils aux quatre
// checkpoints du banc de référence. L'écart se referme-t-il, se stabilise-t-il ou s'agrandit-il ?
println!("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 2 : lever le gel — un plancher d'écart-type change-t-il la donne ?

Le bare-bones **gèle** au meilleur global : quand `ancre == gbest`, l'écart-type s'annule et
la particule-élite ne peut plus bouger. Utilisez le paramètre `std_floor` de l'opérateur Rust
et mesurez les trois bras à `std_floor` ∈ {0 (statu quo), 0,25, 0,5} : le gel est-il une
propriété qui *porte* la convergence, ou un frein que l'on lève gratuitement ?

**Indice** : le paramètre est transmis par `solve` à l'opérateur d'échantillonnage — pensez à documenter le
changement de sémantique dans le commentaire de l'expérience, comme le fait le portage MGS pour
son ancre courante. Gardez les mêmes graines, la même population et le même nombre de générations
pour chaque valeur du plancher.

In [10]:
// Exercice 2 — plancher d'écart-type (std_floor).
// Dans la cellule 12, `solve` transmet déjà le plancher à `propose`. Pour chaque gène,
// l'opérateur du cours conserve la moyenne de Kennedy et borne l'écart-type par le bas :
// let mean = (a + g) / 2.0;
// let std = (g - a).abs().max(std_floor);
// let sampled = sample_normal(rng, mean, std)?.clamp(1.0, 10.0);
// Ce rappel se lit dans `propose` ; ne l'ajoutez pas au bloc de courses ci-dessous.
// Dans une copie de la cellule 12, insérez le balayage dans la fermeture `result`,
// juste avant `Ok(())`. La valeur 0.0 rejoue le statu quo avec les mêmes graines.
// for floor in [0.0_f64, 0.25, 0.5] {
//     for (arm, label) in [(Arm::Kernel, "noyau"), (Arm::Aligned, "port-aligné"), (Arm::Literal, "port-littéral")] {
//         for seed in SEEDS {
//             let run = solve(arm, seed, POPULATION, GENERATIONS, floor)?;
//             println!("{label}, plancher {floor:.2}, graine {seed} : {} conflits, checkpoints {:?}",
//                 run.best.conflicts, run.checkpoints);
//         }
//     }
// }
// À compléter : comparez chaque plancher à 0.0 à bras et graine identiques,
// puis expliquez ce que les trajectoires permettent de conclure sur le gel.
println!("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 3 : profiler la fitness — où va la milliseconde ?

Le rapport ms/éval du banc mêle le tirage gaussien, la correction de bornes et le coût Sudoku. Comme
dans les paires précédentes, séparez le **décodage** du **comptage de conflits** sur les 500
vecteurs témoins, pour les deux implémentations Rust : une voie reste-t-elle plus rapide,
et la part du décodage et du comptage diffère-t-elle d'une voie à l'autre ?

In [11]:
// Exercice 3 — profiler le décodage et le comptage sur les 500 vecteurs communs.
// Dans une copie de la cellule 13, insérez ce bloc dans la fermeture `result`,
// après le contrôle d'accord des coûts et avant le chronométrage de la fitness complète.
// fn profile<G>(
//     vectors: &[Vec<f64>],
//     decode: impl Fn(&[f64]) -> Result<G, &'static str>,
//     cost: impl Fn(&G) -> usize,
// ) -> Result<(f64, f64), &'static str> {
//     let mut grids = Vec::with_capacity(vectors.len());
//     let start = Instant::now();
//     for vector in vectors {
//         grids.push(decode(black_box(vector.as_slice()))?);
//     }
//     let decode_ms = start.elapsed().as_secs_f64() * 1_000.0;
//     let start = Instant::now();
//     for grid in &grids {
//         black_box(cost(black_box(grid)));
//     }
//     let cost_ms = start.elapsed().as_secs_f64() * 1_000.0;
//     Ok((decode_ms, cost_ms))
// }
// let (decode_a_ms, cost_a_ms) = profile(&vectors, decode_a, cost_a)?;
// let (decode_b_ms, cost_b_ms) = profile(&received, decode_b, cost_b)?;
// À compléter : répétez cinq fois par voie, prenez la médiane de chaque phase
// et rapportez-la en ms/vecteur. Comparez la part de chaque phase entre les deux voies.
println!("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Note de parité et de reproductibilité

**Protocole.** Même grille Easy[0], même représentation R1, même fonction de coût en conflits, mêmes graines `{0, 1, 7, 42}`, même population de 50 et mêmes 160 générations. Les vecteurs LCG de contrôle ont 51 composantes ; seuls les 36 gènes associés aux cases vides sont décodés.

**Parité exacte.** Les contrôles comparent à tolérance nulle les dimensions de la grille, les coûts des trois témoins LCG (`67, 71, 60`), le tirage gaussien dégénéré et les budgets (8 000 évaluations pour le noyau, 8 050 pour chacun des ports). Ils vérifient aussi l'accord du re-comptage sur les meilleures solutions, l'écart nul entre les ports aux checkpoints et la répétabilité des 12 couples bras–graine. Les temps sont exclus du test de répétabilité.

**Parité sous tolérance.** `StdRng::seed_from_u64` initialise le générateur avant la création de chaque population. À versions et ordre des tirages fixés, une répétition rejoue la même suite Rust ; la même graine ne recrée ni la suite .NET ni celle de NumPy. Les scores, leurs médianes et leurs extrêmes, ainsi que les checkpoints du noyau, sont comparés aux sorties du carnet d'origine sous une tolérance absolue de 12 conflits. Les écarts sont calculés et affichés, pas déduits du choix de la graine.

**Durées.** Ni les millisecondes ni les rapports de temps ne sont des critères de parité : ils dépendent de la machine, du compilateur et des implémentations chronométrées.

## Références et provenance

- James Kennedy, *Bare Bones Particle Swarms*, IEEE Swarm Intelligence Symposium, 2003.
- [Carnet d'origine — Bare Bones PSO MGS contre mealpy](MGS-28-BareBonesPSO-vs-Mealpy.ipynb) : composé MetaGeneticSharp/GeneticSharp et sous-classe de `Optimizer` dans mealpy 3.0.2, exécutés via PythonNet dans le même noyau .NET.
- Le port Rust écrit directement l'opérateur BBPSO pour rendre visibles l'ancre, la sélection et la gestion de la graine. Il n'utilise ni pont Python ni le crate `genetic_algorithm`.

## À retenir

- **Les deux variantes du port peuvent donner les mêmes meilleurs coûts sans suivre les mêmes positions courantes.** Avec la même population initiale et des améliorations strictes, la position courante du port aligné joue le même rôle que le `pbest_i` du port littéral. Le banc vérifie l'écart d'ancrage sur les scores finaux et aux quatre checkpoints, graine par graine.
- **L'écart de noyau se lit graine par graine.** Le noyau et le port aligné utilisent la même ancre `x_i`, mais le best-N sélectionne dans le lot des parents et des enfants, tandis que le port aligné décide séparément pour chaque particule. L'initialisation n'est pas non plus comptée de la même manière. Le classement par médiane résume ces différences sans en isoler une cause.
- **Un écart-type nul fige la coordonnée à sa moyenne**, pas toute la population. La sélection élitiste du noyau conserve le meilleur coût atteint, même si les nouveaux tirages n'améliorent rien.
- **La graine rend une course rejouable dans la même configuration**, pas interchangeable entre bibliothèques. Le banc vérifie la répétabilité en Rust et compare les scores de la source sous tolérance.
- **Il n'y a pas de BBPSO natif dans le catalogue de mealpy 3.0.2** (`OriginalPSO`, `P_PSO`, `C_PSO`, `CL_PSO`, `AIW_PSO`, `HPSO_TVAC`, `LDW_PSO`) : le bras mealpy est une sous-classe de `Optimizer`, tandis que Rust implémente directement les variantes.

## Vérifiez votre compréhension

1. Les deux variantes du port donnent-elles le même résultat ?
   <details><summary>Réponse</summary>

   Le banc compare les conflits finaux et les checkpoints sur les quatre graines. Le port aligné n'accepte qu'une amélioration stricte : sa position courante joue donc le même rôle que le `pbest_i` du port littéral. À tirages et départages identiques, l'écart attendu est nul sur les meilleurs coûts, même si les positions courantes diffèrent.

   </details>

2. L'écart entre le noyau et le port aligné est-il alors un écart d'ancrage ?
   <details><summary>Réponse</summary>

   Non, le banc le nomme écart de **noyau**. Les deux bras ont la même ancre, mais le noyau garde les meilleurs individus du lot parents + enfants, tandis que le port aligné décide pour chaque particule. Les bras ne comptent pas non plus l'initialisation de la même façon. L'écart ne mesure donc pas l'effet isolé d'une seule règle.

   </details>

3. Sur quelles graines l'écart de noyau est-il net ?
   <details><summary>Réponse</summary>

   Lisez, pour chaque graine, les quatre différences « noyau − port aligné » dans la sortie du banc. Un écart est **persistant** si les quatre différences ont le même signe non nul. Un signe négatif indique que le noyau a moins de conflits. La médiane finale, à elle seule, ne permet pas de distinguer un écart persistant d'un croisement ou d'une refermeture.

   </details>

4. Pourquoi le bras mealpy est-il un subclass ?
   <details><summary>Réponse</summary>

   Parce que mealpy 3.0.2 n'expose aucun PSO bare-bones : le catalogue PSO de la bibliothèque n'en contient aucun. Les deux variantes du banc d'origine (`'kennedy'` et `'mgs'`) écrivent donc explicitement la mise à jour gaussienne sans vitesse. Le port Rust reprend cet opérateur sans passer par mealpy et rend visibles l'ancre et la règle de remplacement.

   </details>

## Pour aller plus loin

**Carnet suivant.** [MGS-29 — GA MGS contre mealpy](MGS-29-GA-vs-Mealpy.ipynb) — la paire suivante du même protocole apparié : la revanche du GA, où les deux moteurs s'accordent à un conflit près.

**Lectures.**

- **La question directrice de la campagne** : à budget d'évaluations égal, les écarts entre la bibliothèque C# et la référence Python sont-ils **systématiques** (coût) ou **spécifiques** (qualité) ?
- **Le protocole apparié** : hérité de [MGS-22](MGS-22-MGS-vs-Mealpy.ipynb) — même grille, mêmes graines, déterminisme vérifié. Ici, le décompte de l'initialisation distingue les budgets du noyau et des ports.
- **La décomposition en axes** : croiser les bras plutôt que comparer les paires, pour distinguer la formule, la sélection et le moteur sans attribuer à un seul facteur un écart qui en mêle plusieurs.
- **Kennedy (2003)** — le PSO « bare bones » sans vitesse, qui ancre le tirage sur la meilleure position personnelle : la référence du port littéral.

**Carnets voisins.**

- **[MGS-28 — Carnet d'origine MGS contre mealpy](MGS-28-BareBonesPSO-vs-Mealpy.ipynb)** — la comparaison C#/.NET et mealpy dont ce carnet reprend la grille et le protocole.
- **[MGS-27 — ForensicBasedInvestigation MGS contre mealpy](MGS-27-ForensicBasedInvestigation-vs-Mealpy.ipynb)** — la paire précédente : un FBI où l'appariement ne peut porter que sur le budget.
- **[MGS-31 — Synthèse croisée MGS contre mealpy](MGS-31-Synthese-Croisee.ipynb)** — la synthèse de la campagne : cette paire y est replacée parmi les neuf.
- **[README de la sous-série](README.md)** — la question directrice, le protocole commun et la navigation de la sous-série.

### Bilan de parité

| Section | Écarts à lire dans les sorties Rust |
|---|---|
| Socle, coût et formules | Comparaisons exactes : dimensions R1 (36 cases vides, 45 indices fixes, 36 gènes), coûts LCG (`67, 71, 60`) et quatre tirages à écart-type nul. Le catalogue mealpy est cité comme référence documentaire. |
| Courses témoins | Écart de budget attendu nul (8 000 pour le noyau, 8 050 pour chacun des ports) et écart nul au re-comptage croisé. Les écarts de scores par rapport aux références .NET et mealpy sont affichés avec la bande de 12 conflits. |
| Banc apparié | Le tableau de conformité donne les écarts mesurés par bras et par graine, ceux des médianes et des extrêmes, ainsi que les écarts des checkpoints du noyau. Les écarts entre les ports et la répétabilité sont vérifiés à l'égalité (écart d'ancrage attendu nul, 12 couples bras–graine stables attendus). |
| Fitness isolée | Accord des coûts vérifié point par point sur 500 vecteurs, avec affichage de l'écart maximal. Les durées et leur rapport ne sont pas comparés aux mesures .NET/mealpy. |
| Exercices | Seul le message d'exercice est affiché ; aucun résultat numérique n'est attendu. |